# Standardization
Scaling Age and EstimatedSalary with `StandardScaler`, and checking what it changes.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import seaborn.objects as so
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# one look for every plot: Latin Modern font, white grid
THEME = {**sns.axes_style("whitegrid"), "font.family": "Latin Modern Roman", "font.size": 13}

## Load the data
400 users of a social network: age, estimated salary, and whether they bought the advertised product.

In [ ]:
df = pd.read_csv("data/Social_Network_Ads.csv")
# keep Age, EstimatedSalary and the target Purchased (drop User ID and Gender)
df = df.iloc[:, 2:]
df.sample(5, random_state=1)

## Split first, then scale

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns="Purchased"),   # inputs: Age, EstimatedSalary
    df["Purchased"],                # output
    test_size=0.3, random_state=0)
X_train.shape, X_test.shape

## StandardScaler

In [ ]:
scaler = StandardScaler()
# fit: learn each column's mean and standard deviation, from the training set only
scaler.fit(X_train)
# transform: apply (x - mean) / std to both sets, with the training numbers
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# what fit learned: the mean and standard deviation of each column
print("mean:", scaler.mean_)
print("std: ", scaler.scale_)

In [ ]:
# transform returns a NumPy array; put the column names back for easier reading
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)
X_train_scaled.head()

In [ ]:
# the first row by hand: (26 - mean) / std should match the table above
age, salary = X_train.iloc[0]
print((age - scaler.mean_[0]) / scaler.scale_[0], (salary - scaler.mean_[1]) / scaler.scale_[1])

## Before and after: describe

In [ ]:
np.round(X_train.describe(), 1)

In [ ]:
np.round(X_train_scaled.describe(), 1)

In [ ]:
# check: every scaled column has mean 0 and standard deviation 1 (ddof=0, as the scaler uses)
assert np.allclose(X_train_scaled.mean(), 0)
assert np.allclose(X_train_scaled.std(ddof=0), 1)

## Effect of scaling: scatter plots

In [ ]:
# put both versions in one long table, so one plot can draw them side by side
both = pd.concat([X_train.assign(stage="Before scaling"), X_train_scaled.assign(stage="After scaling")],
                 ignore_index=True)
(so.Plot(both, x="Age", y="EstimatedSalary")
   .facet(col="stage").share(x=False, y=False)
   .add(so.Dot(pointsize=4, alpha=0.6))
   .layout(size=(10, 4)).theme(THEME))

## Effect of scaling: density curves (KDE)

In [ ]:
# long format: one row per value, with the column name in 'feature'
long = both.melt(id_vars="stage", var_name="feature", value_name="value")
(so.Plot(long, x="value", color="feature")
   .facet(col="stage").share(x=False, y=False)
   .add(so.Line(linewidth=2), so.KDE())
   .layout(size=(10, 4)).theme(THEME))

## The shape of each distribution does not change

In [ ]:
(so.Plot(long, x="value")
   .facet(col="stage", row="feature").share(x=False, y=False)
   .add(so.Line(linewidth=2), so.KDE())
   .layout(size=(10, 6)).theme(THEME))

## Why scaling matters: logistic regression
`solver="sag"` trains by small gradient steps, like gradient descent. On raw data it does not reach the best answer in its default 100 steps.

In [ ]:
lr = LogisticRegression(solver="sag", random_state=0)
lr_scaled = LogisticRegression(solver="sag", random_state=0)
lr.fit(X_train, y_train)                 # raw data (a ConvergenceWarning appears here)
lr_scaled.fit(X_train_scaled, y_train)   # scaled data
print("raw   ", accuracy_score(y_test, lr.predict(X_test)))
print("scaled", accuracy_score(y_test, lr_scaled.predict(X_test_scaled)))

In [ ]:
# the default solver (lbfgs) reaches the answer on raw data too, but needs far more steps
for name, X in [("raw", X_train), ("scaled", X_train_scaled)]:
    m = LogisticRegression().fit(X, y_train)
    print(name, "steps:", m.n_iter_[0])

## A decision tree does not care

In [ ]:
dt = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)
dt_scaled = DecisionTreeClassifier(random_state=0).fit(X_train_scaled, y_train)
print("raw   ", accuracy_score(y_test, dt.predict(X_test)))
print("scaled", accuracy_score(y_test, dt_scaled.predict(X_test_scaled)))

## Effect of outliers

In [ ]:
# add three made-up extreme users (df.append was removed in pandas 2.0, so we use pd.concat)
extra = pd.DataFrame({"Age": [5, 90, 95], "EstimatedSalary": [1000, 250000, 350000], "Purchased": [0, 1, 1]})
df_out = pd.concat([df, extra], ignore_index=True)
df_out.shape

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df_out.drop(columns="Purchased"), df_out["Purchased"], test_size=0.3, random_state=0)
scaler = StandardScaler().fit(X_train)
X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns)
# which of the three outliers landed in the training set?
X_train[X_train.index >= 400]

In [ ]:
# mark the added rows (index 400 and up) so they stand out
point = np.where(X_train.index >= 400, "outlier", "normal")
both = pd.concat([X_train.assign(stage="Before scaling", point=point),
                  X_train_scaled.assign(stage="After scaling", point=point)], ignore_index=True)
(so.Plot(both, x="Age", y="EstimatedSalary", color="point")
   .facet(col="stage").share(x=False, y=False)
   .add(so.Dot(pointsize=5))
   .layout(size=(10, 4)).theme(THEME))

The outliers are just as far from the rest after scaling: standardization does not remove them.